In [5]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from transformers import DataCollatorWithPadding


In [6]:
# 1. Load and Prepare Data
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)

# 1. Fill any NaNs that appeared during the heavy cleaning
df['text'] = df['text'].fillna("")

# 2. (Optional but recommended) Remove rows that became empty strings 
# after masking/cleaning so they don't confuse the model
df = df[df['text'].str.strip() != ""]

df.rename(columns={'label': 'labels'}, inplace=True)

df['labels'] = df['labels'].map({'real': 0, 'fake': 1})

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split into 80% training and 20% testing FIRST
split_datasets = hf_dataset.train_test_split(test_size=0.2, seed=42)

# 2. Tokenization Setup
# Removed the conflicting AutoTokenizer lines that referenced an undefined model_id
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512 # DistilBERT cannot handle 1024 tokens
    )

# Apply tokenization to the already-split datasets
tokenized_datasets = split_datasets.map(tokenize_function, batched=True)

Map: 100%|██████████| 3000/3000 [00:00<00:00, 13691.63 examples/s]


In [9]:
print(df['labels'].unique())

<ArrowStringArray>
['fake', 'real']
Length: 2, dtype: str


In [7]:
# Load DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# Training settings (original style)
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # Updated
    eval_dataset=tokenized_datasets["test"],   # Updated
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4490.30it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Train model
trainer.train()

/Users/jackfogerty/Documents/Coding/FakeNewsDetection/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`labels` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [1]:
#LOADING MODEL, PRETRAINED
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "./modernBERT-final"

my_model = AutoModelForSequenceClassification.from_pretrained(save_path)
my_tokenizer = AutoTokenizer.from_pretrained(save_path)

c:\VSCode Python\FakeNewsDetection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 11500.15it/s]


In [3]:
test_results = trainer.predict(tokenized_datasets["test"])

# The rest of the code remains exactly the same
predicted_labels = np.argmax(test_results.predictions, axis=-1)
actual_labels = test_results.label_ids

print(classification_report(actual_labels, predicted_labels, target_names=["real", "fake"]))

NameError: name 'trainer' is not defined

In [12]:
save_path = "./distilBERT-final"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.06it/s]


('./distilBERT-final\\tokenizer_config.json',
 './distilBERT-final\\tokenizer.json')

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(device)

def predict_article(text):
    inputs = my_tokenizer(
        text, 
        return_tensors="pt",
        truncation=True, 
        max_length=512
    )
    
    inputs = {key: value.to(device) for key, value in inputs.items()}

    my_model.eval() 
    with torch.no_grad():
        outputs = my_model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, axis=-1).item()
    
    label_map = {0: "real", 1: "fake"}
    final_prediction = label_map[predicted_id]
    
    return final_prediction

real_text = "Popular streamer Clavicular framemogged by ASU frat leader."
prediction = predict_article(real_text)

print(f"The model predicts the real text as: {prediction}")

fake_text = "Scientists discover that the moon is made out of cheese."
prediction = predict_article(fake_text)

print(f"The model predicts the fake text as: {prediction}")

The model predicts the real text as: real
The model predicts the fake text as: fake
